# MGMT298D: Science and Strategy of AI
## Assignment 7 - Transformers and Attention
### Application: Text Classification and Generation

---

**Instructions:** Complete the exercises by filling in the `???` placeholders and answering the questions. Run all code cells in order.

## Setup and Data Loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from keras import layers
from keras.utils import pad_sequences

np.random.seed(42)
tf.random.set_seed(42)

VOCAB_SIZE = 10000
MAX_LEN = 200

# Load IMDb dataset for sentiment classification
(x_train, y_train), (x_test, y_test) = keras.datasets.imdb.load_data(num_words=VOCAB_SIZE)
x_train = pad_sequences(x_train, maxlen=MAX_LEN)
x_test = pad_sequences(x_test, maxlen=MAX_LEN)

print(f"Training: {x_train.shape} | Test: {x_test.shape}")

## Part 1: Understanding Self-Attention

Self-attention allows the model to weigh the importance of different words when processing each word. Let's visualize how attention works.

In [ ]:
def simple_attention(query, key, value):
    """Simplified self-attention mechanism."""
    # Step 1: Compute attention scores (how much each word attends to others)
    scores = np.dot(query, key.T)
    
    # Step 2: Normalize with softmax (so weights sum to 1)
    weights = np.exp(scores) / np.sum(np.exp(scores), axis=-1, keepdims=True)
    
    # Step 3: Weighted sum of values
    output = np.dot(weights, value)
    return output, weights

# Simple example: 4 words, each represented by 3-dim vector
words = np.array([[1, 0, 1],   # Word 1
                  [0, 1, 0],   # Word 2
                  [1, 1, 0],   # Word 3
                  [0, 0, 1]])  # Word 4

output, attention_weights = simple_attention(words, words, words)

# Visualize attention weights
plt.figure(figsize=(6, 5))
plt.imshow(attention_weights, cmap='Blues')
plt.colorbar(label='Attention Weight')
plt.xlabel('Key (attending to)')
plt.ylabel('Query (from)')
plt.title('Self-Attention Weights')
plt.xticks(range(4), ['Word 1', 'Word 2', 'Word 3', 'Word 4'])
plt.yticks(range(4), ['Word 1', 'Word 2', 'Word 3', 'Word 4'])
plt.tight_layout()
plt.show()

print("Attention weights (each row sums to 1):")
print(attention_weights.round(3))

## Part 2: Transformer Components

A transformer consists of: (1) Token + Position Embeddings, (2) Multi-Head Attention, (3) Feed-Forward Network.

In [ ]:
class TokenAndPositionEmbedding(layers.Layer):
    """Combines word embeddings with position information."""
    def __init__(self, maxlen, vocab_size, embed_dim):
        super().__init__()
        self.token_emb = layers.Embedding(input_dim=vocab_size, output_dim=embed_dim)
        self.pos_emb = layers.Embedding(input_dim=maxlen, output_dim=embed_dim)

    def call(self, x):
        maxlen = tf.shape(x)[-1]
        positions = tf.range(start=0, limit=maxlen, delta=1)
        positions = self.pos_emb(positions)
        x = self.token_emb(x)
        return x + positions  # Add position info to word embeddings


class TransformerBlock(layers.Layer):
    """Single transformer block with attention and feed-forward layers."""
    def __init__(self, embed_dim, num_heads, ff_dim, dropout_rate=0.1):
        super().__init__()
        self.att = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = keras.Sequential([
            layers.Dense(ff_dim, activation="relu"),
            layers.Dense(embed_dim)
        ])
        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = layers.Dropout(dropout_rate)
        self.dropout2 = layers.Dropout(dropout_rate)

    def call(self, inputs, training=False):
        # Self-attention with residual connection
        attn_output = self.att(inputs, inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        
        # Feed-forward with residual connection
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

print("Transformer components defined")

## Part 3: Effect of Number of Attention Heads

Multi-head attention allows the model to attend to information from different representation subspaces.

In [ ]:
EMBED_DIM = 64
FF_DIM = 64

def build_transformer(num_heads):
    inputs = layers.Input(shape=(MAX_LEN,))
    x = TokenAndPositionEmbedding(MAX_LEN, VOCAB_SIZE, EMBED_DIM)(inputs)
    x = TransformerBlock(EMBED_DIM, num_heads, FF_DIM)(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(32, activation="relu")(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)
    
    model = keras.Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return model

# Test different numbers of attention heads
head_configs = [1, 2, 4, 8]
head_results = {"num_heads": [], "test_accuracy": [], "params": []}

for num_heads in head_configs:
    print(f"Training with {num_heads} attention head(s)...")
    model = build_transformer(num_heads)
    model.fit(x_train, y_train, epochs=3, batch_size=128, 
              validation_split=0.15, verbose=0)
    
    test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
    head_results["num_heads"].append(num_heads)
    head_results["test_accuracy"].append(test_acc)
    head_results["params"].append(model.count_params())

head_df = pd.DataFrame(head_results)
print("\n=== Attention Heads Results ===")
print(head_df.to_string(index=False))

## Part 4: Effect of Transformer Depth

Stacking multiple transformer blocks allows the model to learn more complex patterns.

In [ ]:
def build_deep_transformer(num_blocks):
    inputs = layers.Input(shape=(MAX_LEN,))
    x = TokenAndPositionEmbedding(MAX_LEN, VOCAB_SIZE, EMBED_DIM)(inputs)
    
    # Stack multiple transformer blocks
    for _ in range(num_blocks):
        x = TransformerBlock(EMBED_DIM, num_heads=4, ff_dim=FF_DIM)(x)
    
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(32, activation="relu")(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)
    
    model = keras.Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return model

# Test different depths
depth_configs = [1, 2, 3, 4]
depth_results = {"num_blocks": [], "test_accuracy": [], "params": []}

for num_blocks in depth_configs:
    print(f"Training with {num_blocks} transformer block(s)...")
    model = build_deep_transformer(num_blocks)
    model.fit(x_train, y_train, epochs=3, batch_size=128,
              validation_split=0.15, verbose=0)
    
    test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
    depth_results["num_blocks"].append(num_blocks)
    depth_results["test_accuracy"].append(test_acc)
    depth_results["params"].append(model.count_params())

depth_df = pd.DataFrame(depth_results)
print("\n=== Transformer Depth Results ===")
print(depth_df.to_string(index=False))

In [ ]:
# Visualize depth vs accuracy trade-off
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(depth_df["num_blocks"], depth_df["test_accuracy"], marker="o", linewidth=2, markersize=10)
ax.set_xlabel("Number of Transformer Blocks")
ax.set_ylabel("Test Accuracy")
ax.set_title("Model Depth vs Performance")
ax.set_xticks(depth_configs)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Part 5: Text Generation with Pre-trained Transformers (GPT-2)

In [ ]:
!pip install transformers -q

from transformers import TFGPT2LMHeadModel, GPT2Tokenizer

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
gpt2_model = TFGPT2LMHeadModel.from_pretrained("gpt2", pad_token_id=tokenizer.eos_token_id)
print("GPT-2 model loaded")

In [ ]:
def generate_text(prompt, temperature=1.0, max_length=50):
    """Generate text with GPT-2."""
    input_ids = tokenizer.encode(prompt, return_tensors='tf')
    output = gpt2_model.generate(
        input_ids, do_sample=True, max_length=max_length,
        temperature=temperature, top_k=50, top_p=0.95
    )
    return tokenizer.decode(output[0], skip_special_tokens=True)

# Compare different temperatures
prompt = "The future of artificial intelligence is"

print(f"Prompt: {prompt}\n")
print("=== Temperature 0.3 (more focused) ===")
print(generate_text(prompt, temperature=0.3))
print("\n=== Temperature 1.0 (balanced) ===")
print(generate_text(prompt, temperature=1.0))
print("\n=== Temperature 1.5 (more creative) ===")
print(generate_text(prompt, temperature=1.5))

In [ ]:
# ============================================================
# EXERCISE: Experiment with your own prompt and settings
# ============================================================
YOUR_PROMPT = ???  # e.g., "A good manager should always"
YOUR_TEMPERATURE = ???  # Try values between 0.2 and 1.5

print(f"Prompt: {YOUR_PROMPT}")
print(f"Temperature: {YOUR_TEMPERATURE}\n")
print(generate_text(YOUR_PROMPT, temperature=YOUR_TEMPERATURE, max_length=75))

## Part 6: Transformer vs RNN Comparison

In [ ]:
# Build LSTM model for comparison
lstm_model = keras.Sequential([
    layers.Input(shape=(MAX_LEN,)),
    layers.Embedding(VOCAB_SIZE, EMBED_DIM),
    layers.LSTM(64),
    layers.Dense(32, activation="relu"),
    layers.Dense(1, activation="sigmoid")
])
lstm_model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

print("Training LSTM model...")
lstm_model.fit(x_train, y_train, epochs=3, batch_size=128, validation_split=0.15, verbose=1)
lstm_loss, lstm_acc = lstm_model.evaluate(x_test, y_test, verbose=0)

# Build best transformer for comparison
transformer_model = build_deep_transformer(num_blocks=2)
print("\nTraining Transformer model...")
transformer_model.fit(x_train, y_train, epochs=3, batch_size=128, validation_split=0.15, verbose=1)
transformer_loss, transformer_acc = transformer_model.evaluate(x_test, y_test, verbose=0)

# Comparison
comparison = pd.DataFrame({
    "Model": ["LSTM", "Transformer (2 blocks)"],
    "Test Accuracy": [lstm_acc * 100, transformer_acc * 100],
    "Parameters": [lstm_model.count_params(), transformer_model.count_params()]
})
print("\n=== Model Comparison ===")
print(comparison.to_string(index=False))

---
## Questions

### Question 1: Self-Attention

**Q1:** In the attention visualization (Part 1), what do the attention weights represent? Why is it useful for a model to "attend" to different parts of the input when processing each word?

*Your answer:*


---
### Question 2: Multi-Head Attention

**Q2a:** How did increasing the number of attention heads affect model performance? Why might multiple heads be beneficial?

*Your answer:*


**Q2b:** GPT-4 uses 96 attention heads, while our small model used 1-8. What are the trade-offs of using more heads in terms of computation, interpretability, and performance?

*Your answer:*


---
### Question 3: Temperature in Text Generation

**Q3:** You experimented with different temperature values for GPT-2. What did you observe? If a company is using AI to generate customer service responses, what temperature would you recommend and why?

*Your answer:*


---
### Question 4: Transformers vs RNNs

**Q4:** Compare the LSTM and Transformer results. Transformers have largely replaced RNNs for NLP tasks. What architectural advantages do transformers have? (Hint: think about how each processes sequences.)

*Your answer:*


---
### Question 5: Business Applications

**Q5:** Name three business applications where transformer-based models (like BERT for classification or GPT for generation) could add value. For each, explain whether you'd use a classifier or generator, and what risks you'd need to manage.

*Your answer:*
